# Tutorial 11: Semantic Segmentation using Pre-Trained Model (DeepLabV3+)

## Objectives
- Understand the architecture of DeepLabV3
- Understanding Semantic Segmentation
- Visualizing the results

We'll use **DeepLabV3** with a ResNet backbone from PyTorch's `torchvision` library for semantic segmentation.

This notebook is ready to run on **Google Colab** (Runtime → Change runtime type → GPU recommended).

## 1. Import Libraries

In [ ]:
import torch
from torchvision import models, transforms
from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import requests
from io import BytesIO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Pretrained DeepLabV3 Model

We load DeepLabV3 with a ResNet-101 backbone pretrained on a COCO subset that uses the 21 Pascal VOC classes (background + 20 object classes).

In [ ]:
# Modern recommended way (weights enum)
weights = DeepLabV3_ResNet101_Weights.DEFAULT  # COCO_WITH_VOC_LABELS_V1
model = models.segmentation.deeplabv3_resnet101(weights=weights)
model = model.to(device)
model.eval()  # Set model to evaluation mode

print("DeepLabV3-ResNet101 loaded successfully!")
print(f"Number of classes: {len(weights.meta['categories'])}")
print("Categories:", weights.meta['categories'])

## 3. Load and Preprocess an Image

You can either:
1. Upload your own image using the cell below, **or**
2. Use a sample image from a URL (default).

In [ ]:
# Option A: Upload your own image (uncomment if needed)
# from google.colab import files
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]
# image = Image.open(image_path).convert("RGB")

# Option B: Download a sample image (person + dog – classic VOC-style scene)
url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")
print(f"Image size: {image.size}")

# Display the original image
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
# Preprocessing pipeline (ImageNet mean/std as required by the model)
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Preprocess the image and add batch dimension
input_tensor = preprocess(image).unsqueeze(0).to(device)
print(f"Input tensor shape: {input_tensor.shape}")

## 4. Perform Semantic Segmentation

In [ ]:
# Perform inference
with torch.no_grad():
    output = model(input_tensor)["out"][0]  # Extract segmentation logits (C, H, W)

# Convert output to class predictions (argmax over class dimension)
output_predictions = output.argmax(0)  # Shape: (H, W)
print(f"Output predictions shape: {output_predictions.shape}")
print(f"Unique predicted classes: {torch.unique(output_predictions).tolist()}")

## 5. Visualize the Segmentation Results

We map each of the 21 Pascal VOC classes to a distinct color.

In [ ]:
# Standard Pascal VOC 21-class color palette
VOC_COLORMAP = np.array([
    [0, 0, 0],        # 0  background
    [128, 0, 0],      # 1  aeroplane
    [0, 128, 0],      # 2  bicycle
    [128, 128, 0],    # 3  bird
    [0, 0, 128],      # 4  boat
    [128, 0, 128],    # 5  bottle
    [0, 128, 128],    # 6  bus
    [128, 128, 128],  # 7  car
    [64, 0, 0],       # 8  cat
    [192, 0, 0],      # 9  chair
    [64, 128, 0],     # 10 cow
    [192, 128, 0],    # 11 diningtable
    [64, 0, 128],     # 12 dog
    [192, 0, 128],    # 13 horse
    [64, 128, 128],   # 14 motorbike
    [192, 128, 128],  # 15 person
    [0, 64, 0],       # 16 pottedplant
    [128, 64, 0],     # 17 sheep
    [0, 192, 0],      # 18 sofa
    [128, 192, 0],    # 19 train
    [0, 64, 128],     # 20 tvmonitor
], dtype=np.uint8)

VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog",
    "horse", "motorbike", "person", "pottedplant", "sheep",
    "sofa", "train", "tvmonitor"
]

def decode_segmentation(mask, colormap=VOC_COLORMAP):
    """Map class indices to RGB colors."""
    r = np.zeros_like(mask, dtype=np.uint8)
    g = np.zeros_like(mask, dtype=np.uint8)
    b = np.zeros_like(mask, dtype=np.uint8)
    for cls_id in range(len(colormap)):
        idx = mask == cls_id
        r[idx] = colormap[cls_id, 0]
        g[idx] = colormap[cls_id, 1]
        b[idx] = colormap[cls_id, 2]
    return np.stack([r, g, b], axis=2)

# Decode the segmentation map
segmentation_map = decode_segmentation(output_predictions.cpu().numpy())

# Plot original image and segmentation side-by-side
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(image)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Semantic Segmentation (DeepLabV3)")
plt.imshow(segmentation_map)
plt.axis("off")

plt.tight_layout()
plt.show()

# Optional: show which classes were detected
unique_classes = torch.unique(output_predictions).cpu().numpy()
print("Detected classes:")
for c in unique_classes:
    print(f"  {c}: {VOC_CLASSES[c]}")

### Overlay visualization (optional but useful)

In [ ]:
# Create a semi-transparent overlay
overlay = Image.fromarray(segmentation_map).convert("RGBA")
original = image.convert("RGBA")
blended = Image.blend(original, overlay, alpha=0.5)

plt.figure(figsize=(8, 6))
plt.imshow(blended)
plt.title("Segmentation Overlay")
plt.axis("off")
plt.show()

---
# Tasks

## Task 1: Test the model on additional / custom images

Try different images (upload your own or change the URL) and observe the segmentation quality.

In [ ]:
def segment_and_visualize(image_url_or_path, model, device, title="Custom Image"):
    """Helper to run segmentation on any image (URL or local path)."""
    if image_url_or_path.startswith("http"):
        response = requests.get(image_url_or_path)
        img = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        img = Image.open(image_url_or_path).convert("RGB")

    input_t = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(input_t)["out"][0]
    pred = out.argmax(0).cpu().numpy()
    seg_map = decode_segmentation(pred)

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title(f"{title} - Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(seg_map)
    plt.title("Segmentation")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    blended = Image.blend(img.convert("RGBA"),
                          Image.fromarray(seg_map).convert("RGBA"), alpha=0.5)
    plt.imshow(blended)
    plt.title("Overlay")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    unique = np.unique(pred)
    print("Detected:", [VOC_CLASSES[c] for c in unique])
    return pred

# Example: another public image
segment_and_visualize(
    "https://images.unsplash.com/photo-1517849845537-4d257902454a?w=640",
    model, device, title="Dog"
)

### Upload your own image (Colab)
Uncomment and run the next cell to upload and segment your own picture.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# for fname in uploaded.keys():
#     print(f"Segmenting {fname}...")
#     segment_and_visualize(fname, model, device, title=fname)

---
## Task 2: Implement Mask R-CNN (Instance Segmentation)

Mask R-CNN performs **instance segmentation** (detects individual object instances + pixel masks) rather than pure semantic segmentation.

We use the pretrained Mask R-CNN with ResNet-50-FPN backbone from `torchvision`.

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.utils import draw_segmentation_masks, draw_bounding_boxes
import torchvision.transforms.functional as F

# Load pretrained Mask R-CNN
mask_weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
mask_model = maskrcnn_resnet50_fpn(weights=mask_weights)
mask_model = mask_model.to(device)
mask_model.eval()

print("Mask R-CNN loaded successfully!")
print("COCO categories (first 20):", mask_weights.meta["categories"][:20], "...")

In [ ]:
# Prepare the same image for Mask R-CNN (different preprocessing)
mask_preprocess = mask_weights.transforms()
mask_input = mask_preprocess(image).to(device)

with torch.no_grad():
    predictions = mask_model([mask_input])[0]

# Filter by confidence score
score_threshold = 0.7
keep = predictions["scores"] > score_threshold
boxes = predictions["boxes"][keep].cpu()
labels = predictions["labels"][keep].cpu()
scores = predictions["scores"][keep].cpu()
masks = predictions["masks"][keep].cpu()  # (N, 1, H, W)

print(f"Detected {len(scores)} instances above threshold {score_threshold}")
for i, (lab, sc) in enumerate(zip(labels, scores)):
    cat_name = mask_weights.meta["categories"][lab]
    print(f"  {i+1}. {cat_name}  (score={sc:.3f})")

In [ ]:
# Visualize Mask R-CNN results
img_tensor = F.pil_to_tensor(image)  # (C, H, W) uint8

# Boolean masks for drawing
bool_masks = masks.squeeze(1) > 0.5  # (N, H, W)

# Draw masks and boxes
result = draw_segmentation_masks(img_tensor, bool_masks, alpha=0.6)
result = draw_bounding_boxes(
    result,
    boxes,
    labels=[mask_weights.meta["categories"][l] for l in labels],
    colors="red",
    width=3
)

plt.figure(figsize=(12, 8))
plt.imshow(result.permute(1, 2, 0).numpy())
plt.title("Mask R-CNN Instance Segmentation")
plt.axis("off")
plt.show()

### Side-by-side comparison: Semantic (DeepLabV3) vs Instance (Mask R-CNN)

In [ ]:
plt.figure(figsize=(16, 6))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(segmentation_map)
plt.title("DeepLabV3 (Semantic)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(result.permute(1, 2, 0).numpy())
plt.title("Mask R-CNN (Instance)")
plt.axis("off")

plt.tight_layout()
plt.show()

print("""
Key difference:
- Semantic Segmentation (DeepLabV3): labels every pixel with a class (e.g. all 'person' pixels are the same).
- Instance Segmentation (Mask R-CNN): distinguishes individual objects of the same class and provides per-instance masks + boxes.
""")

---
## Summary

| Model              | Type                  | Backbone       | Dataset (pretrain)      | Output                          |
|--------------------|-----------------------|----------------|-------------------------|---------------------------------|
| DeepLabV3          | Semantic Segmentation | ResNet-101     | COCO (VOC labels)       | Per-pixel class map (21 classes)|
| Mask R-CNN         | Instance Segmentation | ResNet-50-FPN  | COCO                    | Boxes + per-instance masks      |

**Tips for Colab**
- Use a GPU runtime for faster inference.
- You can upload your own images with `files.upload()`.
- Experiment with different score thresholds for Mask R-CNN.
- Try other backbones: `deeplabv3_resnet50`, `deeplabv3_mobilenet_v3_large`.